 
Add Input `himanshuwagh/spotify-million`, dừng lần chạy cũ và dọn checkpoint cũ trước khi chạy.
Chạy cell mã bên dưới. Kết quả: `/kaggle/working/music_clean_compact`. Xử lý toàn bộ slice, vẫn kiểm tra Jaccard > 90%.
Bản này dùng Parquet ZSTD tạm và tự dọn sau khi dùng; chưa được đo trên toàn bộ MPD thật. Shuffle vẫn cần đĩa trống.
Không cần GPU. Dùng PySpark đã có và tương thích với kernel; không chạy cell mã cũ cùng lúc.


| Bước | Xử lý trong code |
|---|---|
| Làm phẳng JSON | Dùng `explode` tách playlist và bài hát thành các dòng |
| Chuẩn hóa văn bản | Xóa khoảng trắng đầu/cuối, gộp khoảng trắng liên tiếp; tên bị thiếu chuyển thành chuỗi rỗng |
| Xử lý số người theo dõi | `num_followers` bị thiếu hoặc âm được chuyển về `0` |
| Lọc dòng không hợp lệ | Loại bài có `pos` thiếu/âm hoặc `track_uri` sai định dạng |
| Loại dòng trùng | Khử trùng theo bộ `(pid, pos, track_uri)` |
| Mã hóa bài hát | Ánh xạ `track_uri` thành `track_id` số nguyên |
| Lọc theo tần suất | Giữ playlist có ít nhất **5 bài khác nhau**, mỗi bài thuộc ít nhất **5 playlist khác nhau** |
| Loại playlist trùng | Loại tập bài giống hệt và playlist có **Jaccard > 90%** |
| Lọc lặp | Lặp các bước lọc đến khi dữ liệu ổn định |
| Kiểm tra kết quả | Tính lại `num_tracks`, kiểm tra khóa và các ngưỡng trước khi xuất Parquet |

In [1]:
import argparse
import json
import sys
import time
import uuid
from fractions import Fraction
from pathlib import Path

from pyspark.sql import SparkSession, functions as F, types as T


DATASET_SOURCE = "https://www.kaggle.com/datasets/himanshuwagh/spotify-million"


TRACK_SCHEMA = T.StructType([
    T.StructField("track_uri", T.StringType()),
    T.StructField("track_name", T.StringType()),
    T.StructField("artist_name", T.StringType()),
    T.StructField("pos", T.IntegerType()),
])
PLAYLIST_SCHEMA = T.StructType([
    T.StructField("pid", T.LongType()),
    T.StructField("name", T.StringType()),
    T.StructField("num_tracks", T.IntegerType()),
    T.StructField("num_followers", T.LongType()),
    T.StructField("tracks", T.ArrayType(TRACK_SCHEMA)),
])
ROOT_SCHEMA = T.StructType([
    T.StructField("playlists", T.ArrayType(PLAYLIST_SCHEMA)),
])

"""In thông báo tiến độ với tiền tố [MPD] và hiển thị ngay trên notebook/terminal."""
def log(message):
    print(f"[MPD] {message}", flush=True)

"""Trả về True nếu DataFrame có ít nhất một dòng; chỉ lấy tối đa một dòng để kiểm tra."""
def has_rows(df):
    return bool(df.take(1))

"""
Báo ValueError nếu DataFrame còn dòng dữ liệu.
Dùng để phát hiện các dòng vi phạm điều kiện kiểm tra đầu vào hoặc đầu ra.
"""
def require_empty(df, message):
    if has_rows(df):
        raise ValueError(message)

"""
Quản lý các bảng Parquet tạm của riêng một lượt chạy.
Lưu dữ liệu có nén và xóa các bản trung gian sau khi dùng để giảm dung lượng đĩa.
"""
class ParquetStages:
   """Khởi tạo bộ quản lý với SparkSession, thư mục tạm và danh sách bảng đang được lưu."""
    def __init__(self, spark, root):
        self.spark, self.root, self.paths = spark, root, {}

    """
    Ghi DataFrame vào thư mục Parquet ZSTD riêng rồi đọc lại và trả về DataFrame mới.
    Đăng ký đường dẫn để có thể xóa sau này; dữ liệu đã ghi giúp ngắt chuỗi tính toán cũ.
    """
    def save(self, df):
        path = self.root + "/" + uuid.uuid4().hex
        df.write.mode("errorifexists").option("compression", "zstd").parquet(path)
        result = self.spark.read.parquet(path)
        self.paths[id(result)] = (result, path)
        return result

    """
    Xóa bản Parquet tạm gắn với chính DataFrame này nếu nó được bộ quản lý đăng ký.
    Chỉ gọi sau khi các bước còn cần đọc dữ liệu đó đã hoàn tất.
    """
    def delete_path(self, path):
        jpath = self.spark._jvm.org.apache.hadoop.fs.Path(path)
        fs = jpath.getFileSystem(self.spark._jsc.hadoopConfiguration())
        if fs.exists(jpath) and not fs.delete(jpath, True):
            raise OSError(f"Khong xoa duoc du lieu tam: {path}")
    """
    Xóa bản Parquet tạm gắn với chính DataFrame này nếu nó được bộ quản lý đăng ký.
    Chỉ gọi sau khi các bước còn cần đọc dữ liệu đó đã hoàn tất.
    """     
    def release(self, df):
        entry = self.paths.pop(id(df), None)
        if entry:
            self.delete_path(entry[1])
            
    #Xóa toàn bộ thư mục tạm của lượt chạy và xóa danh sách bảng đã đăng ký
    def close(self):
        self.paths.clear()
        self.delete_path(self.root)

"""
Lưu một mốc dữ liệu đã tính xong và trả về DataFrame tương ứng.
Trong pipeline, dùng Parquet ZSTD do ParquetStages quản lý; ngoài pipeline, dùng checkpoint.
Nhánh checkpoint cần SparkContext được cấu hình thư mục checkpoint trước.
"""
def snapshot(df):
    store = getattr(df.sparkSession, "_mpd_stages", None)
    return store.save(df) if store else df.checkpoint(eager=True)

"""
Yêu cầu bộ quản lý của SparkSession xóa bản Parquet tạm gắn với DataFrame.
Nếu không có bộ quản lý đang hoạt động thì không thực hiện thao tác nào.
"""
def release(df):
    store = getattr(df.sparkSession, "_mpd_stages", None)
    if store:
        store.release(df)

"""
Tạo biểu thức làm sạch cột văn bản: thay null bằng chuỗi rỗng,
gộp các ký tự khoảng trắng liên tiếp và bỏ khoảng trắng ở hai đầu.
"""
def clean_text(column):
    return F.trim(F.regexp_replace(F.coalesce(column, F.lit("")), r"\s+", " "))


"""
Đọc các slice MPD JSON, quét cả thư mục con và làm phẳng playlist/bài hát bằng explode.
Chuẩn hóa tên, xử lý số người theo dõi thiếu/âm, kiểm tra pid và loại dòng bài không hợp lệ.
Khử trùng (pid, pos, track_uri); một bài lặp ở các vị trí khác nhau vẫn được giữ.
Trả về metadata playlist, các dòng bài hợp lệ, các lượt bài đã khử trùng và thống kê đầu vào.
"""
def load_mpd(spark, input_path):
    source = (spark.read.schema(ROOT_SCHEMA).option("multiLine", True)
              .option("recursiveFileLookup", True)
              .option("pathGlobFilter", "mpd.slice.*.json")
              .option("mode", "FAILFAST").json(input_path))
    file_count = len(source.inputFiles())
    if file_count == 0:
        raise FileNotFoundError("Khong tim thay mpd.slice.*.json; kiem tra input va giai nen du lieu.")
    require_empty(source.filter(F.col("playlists").isNull()),
                  "JSON khong co mang playlists hoac sai cau truc MPD.")
    # Never checkpoint the large nested raw/exploded JSON structures.
    raw = source.select(F.explode("playlists").alias("p")).select("p.*")
    metadata = snapshot(raw.select(
        "pid", clean_text(F.col("name")).alias("name"),
        F.greatest(F.coalesce("num_followers", F.lit(0)), F.lit(0)).alias("num_followers"),
    ))
    require_empty(metadata.filter(F.col("pid").isNull() | (F.col("pid") < 0)),
                  "pid phai la so nguyen khong am.")
    require_empty(metadata.groupBy("pid").count().filter("count > 1"),
                  "Trung pid trong input. Kiem tra cac slice bi nap lap.")
    input_rows = raw.agg(F.coalesce(F.sum(F.when(F.col("tracks").isNotNull(),
        F.size("tracks")).otherwise(0)), F.lit(0)).alias("n")).first().n
    exploded = raw.select("pid", F.explode("tracks").alias("t")).select(
        "pid", F.col("t.pos").alias("pos"),
        F.trim(F.col("t.track_uri")).alias("track_uri"),
        clean_text(F.col("t.track_name")).alias("track_name"),
        clean_text(F.col("t.artist_name")).alias("artist_name"),
    )
    valid = snapshot(exploded.filter(
        F.col("pos").isNotNull() & (F.col("pos") >= 0)
        & F.col("track_uri").rlike(r"^spotify:track:[A-Za-z0-9]+$")
    ))
    # Repeated songs at different positions remain in playlist_tracks.
    require_empty(valid.groupBy("pid", "pos").agg(
        F.countDistinct("track_uri").alias("n")).filter("n > 1"),
        "Mot (pid, pos) co nhieu track_uri khac nhau.")
    events = valid.select("pid", "pos", "track_uri").dropDuplicates()
    valid_count = valid.count()
    stats = {
        "slice_files": file_count,
        "input_playlists": metadata.count(),
        "input_track_rows": input_rows,
        "invalid_track_rows_removed": input_rows - valid_count,
        "duplicate_position_rows_removed": valid_count - events.count(),
    }
    return metadata, valid, events, stats


# Sắp xếp phân tán rồi đánh số, tránh cửa sổ row_number toàn cục.
# ID ổn định với cùng tập URI; thêm/bớt URI ở nguồn có thể làm đổi ID.
"""
Tạo danh mục bài hát và ánh xạ mỗi track_uri sang track_id int32 bắt đầu từ 1.
Sắp URI rồi dùng zipWithIndex để ID tái lập được khi chạy lại cùng tập URI đầu vào.
Nếu metadata của một URI không thống nhất, chọn cặp tên bài/nghệ sĩ nhỏ nhất theo thứ tự sắp xếp.
Trả về DataFrame gồm track_id, track_uri, track_name và artist_name.
"""
def build_track_mapping(spark, valid):
    catalog = snapshot(valid.groupBy("track_uri").agg(F.min(F.struct(
        "track_name", "artist_name")).alias("m")).select("track_uri", "m.*"))
    if catalog.count() > 2147483647:
        raise ValueError("Vuot gioi han int32 cho track_id.")
    indexed = catalog.orderBy("track_uri").rdd.zipWithIndex().map(
        lambda pair: (int(pair[1]) + 1, pair[0]["track_uri"],
                      pair[0]["track_name"], pair[0]["artist_name"]))
    schema = "track_id int, track_uri string, track_name string, artist_name string"
    result = snapshot(spark.createDataFrame(indexed, schema))
    release(catalog)
    return result

"""
Lọc lặp để mỗi playlist có đủ số bài khác nhau và mỗi bài thuộc đủ số playlist khác nhau.
Ngưỡng mặc định là 5/5; lượt lặp bài trong cùng playlist không làm tăng tần suất playlist.
Dừng khi số dòng không đổi; trả về dữ liệu còn lại và lịch sử các vòng lọc.
Báo lỗi nếu chưa ổn định sau max_rounds vòng.
"""
def filter_core(events, min_playlist_tracks=5, min_track_playlists=5, max_rounds=100):
    """Bipartite k-core using DISTINCT playlist-song edges, to a fixed point."""
    current = events
    history = []
    for step in range(1, max_rounds + 1):
        before = current.count()
        good_tracks = (current.select("pid", "track_id").distinct()
                       .groupBy("track_id").count()
                       .filter(F.col("count") >= min_track_playlists).select("track_id"))
        narrowed = current.join(good_tracks, "track_id", "left_semi")
        good_playlists = (narrowed.groupBy("pid").agg(
            F.countDistinct("track_id").alias("n"))
            .filter(F.col("n") >= min_playlist_tracks).select("pid"))
        updated = snapshot(narrowed.join(good_playlists, "pid", "left_semi"))
        after = updated.count()
        history.append({"round": step, "rows_before": before, "rows_after": after})
        log(f"core {step}: {before:,} -> {after:,} dong")
        release(current)
        current = updated
        if before == after:
            return current, history
    raise RuntimeError("Loc tan suat chua hoi tu; tang --max-core-rounds.")

"""
Gom các track_id khác nhau của từng pid thành một mảng đã sắp xếp.
Tập bài này được dùng để so sánh playlist, không xét vị trí hoặc số lần lặp bài.
"""
def playlist_sets(events):
    return events.groupBy("pid").agg(F.sort_array(F.collect_set("track_id")).alias("items"))

"""
Tìm playlist có tập bài giống hệt nhau và chọn pid nhỏ nhất làm đại diện ở bước này.
Trả về các pid cần loại cùng matched_lower_pid; hàm chỉ xác định, chưa xóa dữ liệu.
"""
def exact_duplicates(sets):
    representatives = sets.groupBy("items").agg(F.min("pid").alias("matched_lower_pid"))
    return (sets.join(representatives, "items")
            .filter(F.col("pid") != F.col("matched_lower_pid"))
            .select("pid", "matched_lower_pid"))


"""
Tìm chính xác các cặp playlist có Jaccard lớn hơn ngưỡng, mặc định > 0.9.
Jaccard = số bài chung / số bài thuộc ít nhất một trong hai playlist.
Sắp bài theo tần suất tăng dần, lọc tiền tố và độ dài để giảm ứng viên rồi kiểm tra giao/hợp.
Trả về pid_a < pid_b, kích thước giao/hợp và Jaccard; cặp bằng đúng ngưỡng không được chọn.
"""
def find_near_pairs(events, threshold="0.9"):
    """Exact Jaccard join using a lossless rare-first prefix filter (strict > t)."""
    ratio = Fraction(str(threshold))
    if not 0 < ratio < 1:
        raise ValueError("threshold phai nam giua 0 va 1.")
    p, q = ratio.numerator, ratio.denominator
    distinct_edges = events.select("pid", "track_id").distinct()
    frequencies = distinct_edges.groupBy("track_id").count().withColumnRenamed("count", "df")
    # Every playlist uses the same global (frequency, track_id) token ordering.
    ordered = snapshot(distinct_edges.join(frequencies, "track_id").groupBy("pid").agg(
        F.sort_array(F.collect_list(F.struct("df", "track_id"))).alias("tokens"))
        .withColumn("items", F.expr("transform(tokens, x -> x.track_id)"))
        .select("pid", "items").withColumn("n", F.size("items")))
    # If J(A,B)>p/q, overlap>p*|A|/q. A prefix of n-floor(p*n/q)
    # tokens must intersect B's corresponding prefix. No random hashing.
    pref = snapshot(ordered.select(
        "pid", "n", F.explode(F.expr(
            f"slice(items, 1, n - CAST((CAST(n AS BIGINT) * {p}) DIV {q} AS INT))"
        )).alias("token")))
    a, b = pref.alias("a"), pref.alias("b")
    candidates = snapshot(a.join(b, (
        (F.col("a.token") == F.col("b.token")) & (F.col("a.pid") < F.col("b.pid"))
        & (q * F.least(F.col("a.n"), F.col("b.n"))
           > p * F.greatest(F.col("a.n"), F.col("b.n")))
    )).select(F.col("a.pid").alias("pid_a"), F.col("b.pid").alias("pid_b")).distinct())
    left = ordered.select(F.col("pid").alias("pid_a"), F.col("items").alias("items_a"))
    right = ordered.select(F.col("pid").alias("pid_b"), F.col("items").alias("items_b"))
    scored = (candidates.join(left, "pid_a").join(right, "pid_b")
              .withColumn("intersection", F.size(F.array_intersect("items_a", "items_b")))
              .withColumn("union", F.size("items_a") + F.size("items_b") - F.col("intersection")))
    # Integer arithmetic distinguishes exactly 90% from above 90%.
    result = snapshot(scored.filter(q * F.col("intersection") > p * F.col("union"))
            .select("pid_a", "pid_b", "intersection", "union",
                    (F.col("intersection") / F.col("union")).alias("jaccard")))
    release(candidates)
    release(pref)
    release(ordered)
    return result


"""
Điều phối lọc tần suất và loại playlist trùng/gần trùng đến khi dữ liệu ổn định.
Với mỗi cặp gần trùng, loại pid lớn hơn; chính sách có thể loại nhiều playlist trong một chuỗi cặp.
Lọc lại sau mỗi lần loại để tiếp tục bảo đảm ngưỡng tần suất và phát hiện cặp mới.
Trả về dữ liệu sạch, nhật ký playlist bị loại và thống kê các vòng xử lý.
"""
def clean_until_stable(spark, events, args):
    audit = spark.createDataFrame([], "pid long, matched_lower_pid long, reason string, round int")
    rounds = []
    current = events
    for outer in range(1, args.max_clean_rounds + 1):
        log(f"Vong lam sach {outer}")
        current, core_history = filter_core(current, args.min_playlist_tracks,
                                             args.min_track_playlists, args.max_core_rounds)
        if not has_rows(current):
            raise ValueError("Khong con du lieu sau loc. Dung corpus lon hon; xem README.")
        sets = snapshot(playlist_sets(current))
        exact = snapshot(exact_duplicates(sets))
        release(sets)
        exact_count = exact.count()
        if exact_count:
            updated = snapshot(current.join(exact.select("pid"), "pid", "left_anti"))
            release(current)
            current = updated
        pairs = find_near_pairs(current, args.near_threshold)
        near = snapshot(pairs.groupBy(F.col("pid_b").alias("pid")).agg(
            F.min("pid_a").alias("matched_lower_pid")))
        near_count = near.count()
        release(pairs)
        for removed, reason, count in [(exact, "exact", exact_count), (near, "near", near_count)]:
            if count:
                updated_audit = snapshot(audit.unionByName(
                    removed.withColumn("reason", F.lit(reason)).withColumn("round", F.lit(outer))))
                release(audit)
                audit = updated_audit
        log(f"Bo {exact_count:,} playlist trung; {near_count:,} playlist gan trung")
        rounds.append({"round": outer, "core": core_history,
                       "exact_removed": exact_count, "near_removed": near_count})
        if exact_count + near_count == 0:
            release(exact)
            release(near)
            return current, audit, rounds
        if near_count:
            updated = snapshot(current.join(near.select("pid"), "pid", "left_anti"))
            release(current)
            current = updated
        release(exact)
        release(near)
    raise RuntimeError("Lam sach chua hoi tu; tang --max-clean-rounds.")

"""
Kiểm tra các khóa duy nhất, khóa ngoại, num_tracks và các ngưỡng tần suất đầu ra.
Báo lỗi nếu phát hiện vi phạm; kiểm tra gần trùng đã được hoàn tất ở vòng làm sạch cuối.
"""
def validate_outputs(playlists, tracks, playlist_tracks, mapping, args):
    require_empty(playlists.groupBy("pid").count().filter("count > 1"), "Trung pid dau ra.")
    require_empty(tracks.groupBy("track_id").count().filter("count > 1"), "Trung track_id.")
    require_empty(mapping.groupBy("track_uri").count().filter("count > 1"), "Trung track_uri.")
    require_empty(playlist_tracks.groupBy("pid", "pos").count().filter("count > 1"),
                  "Trung (pid,pos) dau ra.")
    require_empty(playlist_tracks.join(playlists, "pid", "left_anti"), "Khoa ngoai pid bi loi.")
    require_empty(playlist_tracks.join(tracks, "track_id", "left_anti"), "Khoa ngoai track_id bi loi.")
    sizes = playlist_tracks.groupBy("pid").agg(
        F.count("*").alias("actual"), F.countDistinct("track_id").alias("unique_tracks"))
    require_empty(sizes.filter(F.col("unique_tracks") < args.min_playlist_tracks),
                  "Con playlist duoi nguong.")
    require_empty(playlists.join(sizes, "pid").filter("num_tracks != actual"), "Sai num_tracks.")
    require_empty(playlist_tracks.select("pid", "track_id").distinct().groupBy("track_id")
                  .count().filter(F.col("count") < args.min_track_playlists),
                  "Con bai xuat hien duoi nguong playlist.")
    # The final clean round already verified all near pairs, with zero removals.

"""
Ghi một đối tượng Python có thể chuyển thành JSON thành một file UTF-8 trên local/HDFS.
Ghi đè file cùng tên nếu đã có và đóng luồng ghi sau khi thực hiện.
"""
def write_json_file(spark, path, value):
    jpath = spark._jvm.org.apache.hadoop.fs.Path(path)
    fs = jpath.getFileSystem(spark._jsc.hadoopConfiguration())
    stream = fs.create(jpath, True)
    try:
        stream.write(bytearray(json.dumps(value, ensure_ascii=False, indent=2).encode("utf-8")))
    finally:
        stream.close()

"""
Tạo bộ quản lý dữ liệu tạm rồi gọi quy trình xử lý chính.
Ngăn hai pipeline dùng chung SparkSession chạy đồng thời qua bộ quản lý này.
Luôn cố dọn thư mục tạm khi kết thúc, kể cả khi quy trình phát sinh lỗi.
"""
def run_pipeline(spark, args):
    if getattr(spark, "_mpd_stages", None) is not None:
        raise RuntimeError("Khong chay hai pipeline dong thoi trong cung SparkSession.")
    store = ParquetStages(spark, args.output.rstrip("/") + "/_staging/" + uuid.uuid4().hex)
    spark._mpd_stages = store
    try:
        return _run_pipeline(spark, args)
    finally:
        spark._mpd_stages = None
        try:
            store.close()
        except Exception as error:
            log(f"Khong don het thu muc tam: {error}")

"""
Thực hiện toàn bộ quy trình: đọc MPD, mã hóa bài, làm sạch và kiểm tra dữ liệu.
Tính lại num_tracks, xuất ba bảng chính cùng track_mapping và dedup_audit dưới dạng Parquet.
Có thể xuất thêm JSONL GZIP cho Hadoop Streaming khi bật export_streaming.
Ghi báo cáo và dấu _SUCCESS, trả về báo cáo; chưa chia train/test hoặc huấn luyện mô hình.
"""
def _run_pipeline(spark, args):
    start = time.monotonic()
    root = args.output.rstrip("/")
    if not root or root in (".", "/"):
        raise ValueError("Can thu muc output rieng.")
    path = spark._jvm.org.apache.hadoop.fs.Path(root)
    fs = path.getFileSystem(spark._jsc.hadoopConfiguration())
    if fs.exists(path) and not args.overwrite:
        raise FileExistsError(f"Output da ton tai: {root}. Chon thu muc moi hoac --overwrite.")
    fs.mkdirs(path)
    # Remove an old completion marker before any overwrite starts.
    for name in ["_SUCCESS", "cleaning_report.json"]:
        fs.delete(spark._jvm.org.apache.hadoop.fs.Path(root + "/" + name), False)
    log("Doc MPD JSON va explode playlists/tracks")
    metadata, valid, events_uri, stats = load_mpd(spark, args.input)
    log(f"Da doc {stats['slice_files']} slice; {stats['input_playlists']:,} playlist")
    catalog = build_track_mapping(spark, valid)
    events = snapshot(events_uri.join(catalog.select("track_uri", "track_id"), "track_uri")
                      .select("pid", "pos", "track_id"))
    release(valid)
    cleaned, audit, rounds = clean_until_stable(spark, events, args)
    used = cleaned.select("track_id").distinct()
    mapping_full = snapshot(catalog.join(used, "track_id", "left_semi"))
    release(catalog)
    mapping = mapping_full.select("track_id", "track_uri")
    tracks = mapping_full.select("track_id", "track_name", "artist_name")
    playlist_tracks = cleaned.select("pid", "pos", "track_id")
    counts = playlist_tracks.groupBy("pid").agg(F.count("*").cast("int").alias("num_tracks"))
    playlists = snapshot(metadata.join(counts, "pid").select(
        "pid", "name", "num_tracks", "num_followers"))
    release(metadata)
    log("Kiem tra khoa, tan suat va so bai")
    validate_outputs(playlists, tracks, playlist_tracks, mapping, args)
    mode = "overwrite" if args.overwrite else "errorifexists"
    for name, df in [("playlists", playlists), ("tracks", tracks),
                     ("playlist_tracks", playlist_tracks), ("track_mapping", mapping),
                     ("dedup_audit", audit)]:
        log(f"Ghi {name}")
        df.write.mode(mode).option("compression", "zstd").parquet(root + "/" + name)
    if args.export_streaming:
        lines = (playlist_tracks.join(mapping, "track_id").select(F.to_json(F.struct(
            "pid", "pos", "track_id", "track_uri")).alias("value")))
        lines.write.mode(mode).option("compression", "gzip").text(root + "/streaming_tracks_jsonl")
    report = {**stats, "dataset_source": DATASET_SOURCE, "input_path": args.input,
              "input_file_pattern": "mpd.slice.*.json",
              "output_playlists": playlists.count(), "output_tracks": tracks.count(),
              "output_playlist_track_rows": playlist_tracks.count(),
              "removed_duplicate_playlists": audit.count(),
              "threshold": args.near_threshold, "comparison": "Jaccard > threshold",
              "min_distinct_tracks_per_playlist": args.min_playlist_tracks,
              "min_distinct_playlists_per_track": args.min_track_playlists,
              "rounds": rounds, "elapsed_seconds": round(time.monotonic() - start, 2),
              "checks_passed": True, "split_created": False,
              "spark_version": spark.version}
    report["temporary_storage"] = "rolling_zstd_parquet"
    write_json_file(spark, root + "/cleaning_report.json", report)
    write_json_file(spark, root + "/_SUCCESS", {"ok": True})
    log(json.dumps(report, ensure_ascii=False, indent=2))
    return report


"""
Khai báo các tham số dòng lệnh: đường dẫn input/output, cấu hình Spark, các ngưỡng lọc,
giới hạn số vòng, tùy chọn xuất JSONL và cho phép ghi đè. Trả về ArgumentParser.
"""
def parser():
    p = argparse.ArgumentParser(description="Nguoi B: lam sach Spotify MPD bang PySpark")
    p.add_argument("--input", required=True,
                   help="Thu muc dataset (tu quet thu muc con) hoac glob mpd.slice.*.json; local/HDFS")
    p.add_argument("--output", required=True, help="Thu muc Parquet moi, local hoac HDFS")
    p.add_argument("--master", default=None, help="Bo trong de dung master cua spark-submit")
    p.add_argument("--shuffle-partitions", type=int, default=64)
    p.add_argument("--min-playlist-tracks", type=int, default=5)
    p.add_argument("--min-track-playlists", type=int, default=5)
    p.add_argument("--near-threshold", default="0.9")
    p.add_argument("--max-core-rounds", type=int, default=100)
    p.add_argument("--max-clean-rounds", type=int, default=100)
    p.add_argument("--export-streaming", action="store_true")
    p.add_argument("--overwrite", action="store_true")
    return p

"""
Đọc danh sách tham số argv (hoặc tham số terminal nếu argv=None) và kiểm tra giá trị.
Tạo SparkSession, chạy pipeline và luôn dừng Spark khi kết thúc.
"""
def main(argv=None):
    args = parser().parse_args(argv)
    if min(args.min_playlist_tracks, args.min_track_playlists, args.shuffle_partitions,
           args.max_core_rounds, args.max_clean_rounds) < 1:
        raise ValueError("Cac tham so dem phai >= 1.")
    if not 0 < Fraction(args.near_threshold) < 1:
        raise ValueError("--near-threshold phai nam giua 0 va 1.")
    builder = SparkSession.builder.appName("MPD-Week1-PersonB")
    if args.master:
        builder = builder.master(args.master)
    spark = (builder.config("spark.sql.shuffle.partitions", args.shuffle_partitions)
             .config("spark.sql.adaptive.enabled", "true").getOrCreate())
    spark.sparkContext.setLogLevel("WARN")
    try:
        run_pipeline(spark, args)
    finally:
        spark.stop()


if __name__ == "__main__":
    if "ipykernel" in sys.modules:
        candidates = [Path("/kaggle/input/datasets/himanshuwagh/spotify-million"),
                      Path("/kaggle/input/spotify-million")]
        data_root = next((p for p in candidates if p.is_dir()), None)
        if data_root is None:
            raise FileNotFoundError("Gan dataset Spotify Million vao notebook Kaggle truoc khi chay.")
        output_root = Path("/kaggle/working/music_clean_compact")
        notebook_args = ["--input", str(data_root), "--output", str(output_root),
                         "--master", "local[2]", "--shuffle-partitions", "32"]
        if output_root.exists() and not (output_root / "_SUCCESS").exists():
            notebook_args.append("--overwrite")
        main(notebook_args)
    else:
        main()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/30 07:22:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


[MPD] Doc MPD JSON va explode playlists/tracks


[MPD] Da doc 1000 slice; 1,000,000 playlist


[MPD] Vong lam sach 1


[3161.243s][warning][gc,alloc] Executor task launch worker for task 1.0 in stage 80.0 (TID 1705): Retried waiting for GCLocker too often allocating 4195848 words


...

[MPD] Ghi tracks


[MPD] Ghi playlist_tracks


[MPD] Ghi track_mapping


[MPD] Ghi dedup_audit


[MPD] {
  "slice_files": 1000,
  "input_playlists": 1000000,
  "input_track_rows": 66346428,
  "invalid_track_rows_removed": 0,
  "duplicate_position_rows_removed": 0,
  "dataset_source": "https://www.kaggle.com/datasets/himanshuwagh/spotify-million",
  "input_path": "/kaggle/input/datasets/himanshuwagh/spotify-million",
  "input_file_pattern": "mpd.slice.*.json",
  "output_playlists": 990000,
  "output_tracks": 591653,
  "output_playlist_track_rows": 63403784,
  "removed_duplicate_playlists": 4989,
  "threshold": "0.9",
  "comparison": "Jaccard > threshold",
  "min_distinct_tracks_per_playlist": 5,
  "min_distinct_playlists_per_track": 5,
  "rounds": [
    {
      "round": 1,
      "core": [
        {
          "round": 1,
          "rows_before": 66346428,
          "rows_after": 63685978
        },
        {
          "round": 2,
          "rows_before": 63685978,
          "rows_after": 63680944
        },
        {
          "round": 3,
          "rows_before": 63680944,
         